# 04 事件式三状态段与收益审计

本 Notebook 读取 03 号 Notebook 生成的五列事件式 CSV，并用同一个原始现货文件补充实际执行日价格和 O2O_H1。它只做冻结后的诊断，不参与候选筛选。

重点观察：
- 原始基础三状态与事件式最终三状态的天数、占比和变化；
- -1、0、1 各状态的连续段数、均值、中位数、单日/两日占比和最长段；
- 每个信号日、每个最终状态段的 O2O_H1 收益、胜率和收益分布；
- 价格曲线上最终三状态和当天信号的位置，便于截图复核。

正式事件语义：只有当天的 entry signal 才能把基础 0 改为 -1/1，后续无新信号的 0 日不延续重标。

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 260)
pd.set_option('display.max_colwidth', 60)

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'pool_registry.py').is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from diagnostic_utils import (
    PHASES, attach_spot_execution_metrics, duration_summary,
    load_event_signal, segment_return_rows, segment_return_summary,
    state_count_table, state_daily_return_summary, state_runs,
)
from runtime_paths import resolve_output_dir, resolve_spot_path

OUTPUT_DIR = resolve_output_dir(PACKAGE_ROOT)
SPOT_PATH = resolve_spot_path()
SIGNAL_PATH = Path(os.environ.get('EVENT_SIGNAL_CSV_PATH', str(OUTPUT_DIR / 'company_event_three_state_signal.csv'))).expanduser().resolve()
signal = load_event_signal(SIGNAL_PATH)
frame, spot_audit = attach_spot_execution_metrics(signal, SPOT_PATH)
print('事件式五列 CSV:', SIGNAL_PATH)
print('唯一现货输入:', SPOT_PATH)
print('执行日期:', frame['date'].min().date(), '->', frame['date'].max().date(), '；行数:', len(frame))
display(frame.head(10))

## 1. 原始基础状态与事件式最终状态的天数变化

In [ ]:
original_counts = state_count_table(frame, 'three_state', 'original_three_state').rename(columns={'days': 'original_days', 'share_pct': 'original_share_pct'})
final_counts = state_count_table(frame, 'final_three_state', 'event_final_three_state').rename(columns={'days': 'final_days', 'share_pct': 'final_share_pct'})
count_table = original_counts[['state', 'original_days', 'original_share_pct']].merge(final_counts[['state', 'final_days', 'final_share_pct']], on='state')
count_table['delta_days_final_minus_original'] = count_table['final_days'] - count_table['original_days']
count_table['share_delta_pct_point'] = count_table['final_share_pct'] - count_table['original_share_pct']
count_table['meaning'] = count_table['state'].map({-1: '负向持仓', 0: '中性', 1: '正向持仓'})
display(count_table[['state', 'meaning', 'original_days', 'final_days', 'delta_days_final_minus_original', 'original_share_pct', 'final_share_pct', 'share_delta_pct_point']].round(3))

period_rows = []
for period, group in frame.groupby('period', sort=False):
    for state, label in [(-1, '负向持仓'), (0, '中性'), (1, '正向持仓')]:
        old_days = int(group['three_state'].eq(state).sum())
        new_days = int(group['final_three_state'].eq(state).sum())
        period_rows.append({
            'period': period, 'state': state, 'meaning': label, 'period_days': len(group),
            'original_days': old_days, 'final_days': new_days,
            'original_share_pct': old_days / len(group) * 100 if len(group) else np.nan,
            'final_share_pct': new_days / len(group) * 100 if len(group) else np.nan,
            'share_delta_pct_point': (new_days - old_days) / len(group) * 100 if len(group) else np.nan,
        })
display(pd.DataFrame(period_rows).round(3))

## 2. 连续段长度、碎片化和全部非零段

In [ ]:
original_runs = state_runs(frame, 'three_state')
final_runs = state_runs(frame, 'final_three_state')
duration_table = pd.concat([
    duration_summary(original_runs, 'original_three_state'),
    duration_summary(final_runs, 'event_final_three_state'),
], ignore_index=True)
display(duration_table.round(3))

print('最终非零持仓段（完整清单）：')
display(final_runs.loc[final_runs['state'].ne(0)].reset_index(drop=True))

fragment = duration_table.loc[duration_table['series'].eq('event_final_three_state')].copy()
print('重点碎片化指标：单日段、两日以内段和中位持仓长度。')
display(fragment[['state', 'segments', 'mean_days', 'median_days', 'one_day_share', 'two_day_share', 'max_days']].round(3))

## 3. 信号日、逐日收益和持仓段收益分布

In [ ]:
event_rows = frame.loc[frame['minus_entry_signal'].eq(1) | frame['plus_entry_signal'].eq(1)].copy()
event_rows['event_side'] = np.select([event_rows['minus_entry_signal'].eq(1), event_rows['plus_entry_signal'].eq(1)], ['0→-1', '0→1'], default='both')
print('所有实际信号日（date 为实际执行日，收益为指数 O2O_H1）：')
display(event_rows[['date', 'event_side', 'three_state', 'final_three_state', 'close_current', 'o2o_h1_bp']].round(3))

print('最终三状态逐日 O2O_H1 收益分布：')
display(state_daily_return_summary(frame, 'final_three_state').round(3))

segment_rows = segment_return_rows(frame, 'final_three_state')
print('最终三状态持仓段复合收益分布：')
display(segment_return_summary(segment_rows).round(3))
print('每个最终非零段的复合收益（完整清单）：')
display(segment_rows.loc[segment_rows['state'].ne(0)].round(3))

phase_return_rows = []
for period, group in frame.groupby('period', sort=False):
    table = state_daily_return_summary(group, 'final_three_state')
    table.insert(0, 'period', period)
    phase_return_rows.append(table)
print('分周期逐日收益和胜率：')
display(pd.concat(phase_return_rows, ignore_index=True).round(3))

## 4. 价格曲线与事件标记（适合截图）

In [ ]:
plot_df = frame.dropna(subset=['close_current']).copy()
colors = {-1: '#d62728', 0: '#7f7f7f', 1: '#2ca02c'}
labels = {-1: '-1 负向', 0: '0 中性', 1: '+1 正向'}
fig, ax = plt.subplots(figsize=(18, 7))
ax.plot(plot_df['date'], plot_df['close_current'], color='#444444', linewidth=1.0, label='CSI500 close', zorder=1)
for state in (-1, 0, 1):
    part = plot_df.loc[plot_df['final_three_state'].eq(state)]
    ax.scatter(part['date'], part['close_current'], s=13, alpha=0.75, color=colors[state], label=labels[state], zorder=2)
minus = plot_df.loc[plot_df['minus_entry_signal'].eq(1)]
plus = plot_df.loc[plot_df['plus_entry_signal'].eq(1)]
ax.scatter(minus['date'], minus['close_current'], marker='v', s=30, color='#8c1d18', label='0→-1 当天信号', zorder=3)
ax.scatter(plus['date'], plus['close_current'], marker='^', s=30, color='#155d9a', label='0→1 当天信号', zorder=3)
for _, row in pd.concat([minus, plus]).dropna(subset=['o2o_h1_bp']).iterrows():
    ax.annotate(f"{row['o2o_h1_bp']:+.1f}bp", (row['date'], row['close_current']), xytext=(0, 7), textcoords='offset points', fontsize=5.5, rotation=90, ha='center', alpha=0.8)
ax.set_title('事件式最终三状态、当天信号与 CSI500 收盘价')
ax.set_xlabel('实际执行日 date')
ax.set_ylabel('CSI500 close')
ax.grid(alpha=0.22)
ax.legend(ncol=5, loc='upper left')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
print('截图建议：保留上方状态/段表、收益表和本图；CSV 与 JSON 已在输出目录保存。')